In [ ]:
get_ipython().system("pip install -q rapidgzip tiktoken wandb")

In [ ]:
import os
from pathlib import Path

import rapidgzip
import torch
from kaggle_secrets import UserSecretsClient

# This notebook only ever runs on Kaggle, on CUDA.
DEVICE_TYPE: str = "cuda"
print(f"Assuming Kaggle CUDA environment (device={DEVICE_TYPE!r}).")


## Paths
INPUT_DIR: str = "/kaggle/input/notebooks/lucasponer/tokenize-fineweb-edu-sample-10bt/FineWeb-edu_sample_10BT"
OUTPUT_DIR: str = "/kaggle/working/"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

TRAIN_SCRIPT_PATH = f"{OUTPUT_DIR}/train.py"

TOKENIZER_STATE_PATH: str = f"{INPUT_DIR}/tokenizer_state.pkl"

TRAIN_TOKENS_GZ_PATH: str = f"{INPUT_DIR}/train.bin.gz"
TRAIN_TOKENS_INDEX_PATH: str = f"{OUTPUT_DIR}/train.rgidx"

VAL_TOKENS_GZ_PATH: str = f"{INPUT_DIR}/val.bin.gz"
VAL_TOKENS_INDEX_PATH: str = f"{OUTPUT_DIR}/val.rgidx"

if not os.path.exists(TRAIN_TOKENS_INDEX_PATH):
    with rapidgzip.open(TRAIN_TOKENS_GZ_PATH, parallelization=os.cpu_count()) as f:
        f.export_index(TRAIN_TOKENS_INDEX_PATH)

if not os.path.exists(VAL_TOKENS_INDEX_PATH):
    with rapidgzip.open(VAL_TOKENS_GZ_PATH, parallelization=os.cpu_count()) as f:
        f.export_index(VAL_TOKENS_INDEX_PATH)

user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = user_secrets.get_secret("WANDB_PROJECT")
os.environ["FEMTOCHAT_PROFILE"] = user_secrets.get_secret("FEMTOCHAT_PROFILE")

# Only a multi-GPU box goes above NUM_PROCS == 1.
NUM_PROCS: int = max(1, torch.cuda.device_count())

In [ ]:
%%writefile {TRAIN_SCRIPT_PATH}
import contextlib
import math
import os
import pickle
import shutil
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import rapidgzip
import tiktoken
import torch
import torch.distributed as dist
import torch.nn.functional as F
import wandb
from torch import nn
from torch.nn.parallel import DistributedDataParallel
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.distributed import DistributedSampler


# This notebook only ever runs on Kaggle, on CUDA.
DEVICE_TYPE: str = "cuda"


## PATHS
INPUT_DIR: str = "/kaggle/input/notebooks/lucasponer/tokenize-fineweb-edu-sample-10bt/FineWeb-edu_sample_10BT"
OUTPUT_DIR: str = "/kaggle/working/"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
TOKENIZER_STATE_PATH: str = f"{INPUT_DIR}/tokenizer_state.pkl"
TRAIN_TOKENS_GZ_PATH: str = f"{INPUT_DIR}/train.bin.gz"
TRAIN_TOKENS_INDEX_PATH: str = f"{OUTPUT_DIR}/train.rgidx"
VAL_TOKENS_GZ_PATH: str = f"{INPUT_DIR}/val.bin.gz"
VAL_TOKENS_INDEX_PATH: str = f"{OUTPUT_DIR}/val.rgidx"
CHECKPOINT_PATH: str = f"{OUTPUT_DIR}/checkpoints/model.pt"
RESUME_PATH: str | None = CHECKPOINT_PATH
TOKENS_DTYPE: type[np.uint16] = np.uint16

# Full parameters and hyperparameters for a real GPU. Override with the
# FEMTOCHAT_PROFILE env var ("full" or "dry") if you want a smaller run.
PROFILE: str = os.environ.get("FEMTOCHAT_PROFILE", "full")

## MODEL
CTX_LEN: int = 1024
EMB_DIM: int = 768
N_HEADS: int = 12
N_LAYERS: int = 10
DROP_RATE: float = 0.0
BIAS: bool = False

USE_AMP: bool = True
AMP_DTYPE: torch.dtype = torch.float16
USE_COMPILE: bool = True
COMPILE_MODE: str = "default"

## TRAINING
LR: float = 5e-4
MIN_LR: float = 5e-5

if PROFILE == "full":
    WARMUP_STEPS: int = 150
    MAX_STEPS: int = 3000
    EVAL_FREQ: int = 200
    EVAL_ITERS: int = 20
    BATCH_SIZE: int = 12
    GRAD_ACCUM_STEPS: int = 30
else:
    WARMUP_STEPS = 20
    MAX_STEPS = 200
    EVAL_FREQ = 50
    EVAL_ITERS = 10
    BATCH_SIZE = 8
    GRAD_ACCUM_STEPS = 4

BETAS: tuple[float, float] = (0.9, 0.95)
WEIGHT_DECAY: float = 0.1

GRAD_CLIP: float = 1.0

## SAMPLING
MAX_SAMPLE_TOKENS: int = 50
TEMPERATURE: float = 0.8
TOP_K: int | None = 40
SEED: int = 123
VAL_PROMPT: str = "The"


@dataclass
class GPTConfig:
    vocab_size: int
    ctx_len: int = CTX_LEN
    emb_dim: int = EMB_DIM
    n_heads: int = N_HEADS
    n_layers: int = N_LAYERS
    drop_rate: float = DROP_RATE
    bias: bool = BIAS


@dataclass
class TokenizerState:
    pat_str: str
    mergeable_ranks: dict[bytes, int]
    special_tokens: dict[str, int]
    eot_token_id: int


class GzipTokens:
    # Array-like wrapper around a rapidgzip random-access handle.
    def __init__(
        self, gz_path: str, index_path: str, parallelization: int, dtype=TOKENS_DTYPE
    ):
        self.gz_path = gz_path
        self.index_path = index_path
        self.parallelization = parallelization
        self.dtype = dtype
        self.itemsize = np.dtype(dtype).itemsize
        self._file = None
        self._len: int | None = None
        # Open eagerly in the parent so length is known before any worker fork.
        self._ensure_open()

    def _ensure_open(self) -> None:
        if self._file is not None:
            return
        self._file = rapidgzip.open(self.gz_path, parallelization=self.parallelization)
        self._file.import_index(self.index_path)
        self._file.seek(0, 2)
        self._len = self._file.tell() // self.itemsize

    def __getstate__(self) -> dict:
        state = self.__dict__.copy()
        state["_file"] = None
        return state

    def __setstate__(self, state: dict) -> None:
        self.__dict__.update(state)

    def __len__(self) -> int:
        if self._len is None:
            self._ensure_open()
        return self._len

    def __getitem__(self, key: slice) -> np.ndarray:
        self._ensure_open()
        start, stop, step = key.indices(self._len)
        assert step == 1, "GzipTokens only supports contiguous slices"
        self._file.seek(start * self.itemsize)
        raw = self._file.read((stop - start) * self.itemsize)
        return np.frombuffer(raw, dtype=self.dtype)

    def close(self) -> None:
        # rapidgzip warns (and can abort the process) if its background
        # threads are still alive at Python interpreter shutdown, so this
        # must be called explicitly rather than left to __del__/GC timing.
        if self._file is not None:
            self._file.close()
            self._file = None


class GPTDataset(Dataset):
    def __init__(self, ids, max_length, stride):
        self.ids = ids
        self.max_length = max_length
        # Lazily indexed: works for a Python list or an np.memmap without
        # materializing every window up front (needed for corpus-scale token files).
        self.starts = range(0, len(ids) - max_length, stride)

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, idx):
        start = self.starts[idx]
        input_chunk = self.ids[start : start + self.max_length]
        target_chunk = self.ids[start + 1 : start + self.max_length + 1]
        return (
            torch.tensor(input_chunk, dtype=torch.long),
            torch.tensor(target_chunk, dtype=torch.long),
        )


def make_dataloader(
    ids,
    batch_size,
    max_length,
    stride,
    shuffle,
    drop_last,
    num_workers=0,
    rank=0,
    world_size=1,
):
    dataset = GPTDataset(ids, max_length, stride)

    # A sampler does its own shuffling and is mutually exclusive with
    # shuffle= on DataLoader, so only build one under DDP (world_size > 1).
    sampler = None
    if world_size > 1:
        sampler = DistributedSampler(
            dataset,
            num_replicas=world_size,
            rank=rank,
            shuffle=shuffle,
            drop_last=drop_last,
        )

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle if sampler is None else False,
        sampler=sampler,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,
    )

    return dataloader


def build_encoding(state: TokenizerState) -> tiktoken.Encoding:
    return tiktoken.Encoding(
        name="dataset_tokenizer",
        pat_str=state.pat_str,
        mergeable_ranks=state.mergeable_ranks,
        special_tokens=state.special_tokens,
    )


def load_tokenizer_state(path: str):
    with open(path, "rb") as f:
        tokenizer_state = pickle.load(f)
    encoding = build_encoding(tokenizer_state)
    return encoding, tokenizer_state.eot_token_id, tokenizer_state


class MultiHeadAttention(nn.Module):
    def __init__(self, dim_in, dim_out, ctx_len, dropout, num_heads, bias=False):
        super().__init__()
        assert dim_out % num_heads == 0, "dim_out must be divisible by n_heads"

        self.dim_out = dim_out
        self.num_heads = num_heads
        self.head_dim = dim_out // num_heads

        self.W_query = nn.Linear(dim_in, dim_out, bias=bias)
        self.W_key = nn.Linear(dim_in, dim_out, bias=bias)
        self.W_value = nn.Linear(dim_in, dim_out, bias=bias)
        self.out_proj = nn.Linear(dim_out, dim_out, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, num_tokens, dim_in = x.shape

        keys = self.W_key(x)  # Shape: (batch_size, num_tokens, dim_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Unroll last dim: (batch_size, num_tokens, dim_out) -> (batch_size, num_tokens, num_heads, head_dim)
        keys = keys.reshape(batch_size, num_tokens, self.num_heads, self.head_dim)
        values = values.reshape(batch_size, num_tokens, self.num_heads, self.head_dim)
        queries = queries.reshape(batch_size, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (batch_size, num_tokens, num_heads, head_dim) -> (batch_size, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        dropout_p = self.dropout.p if self.training else 0.0
        out_embeds = F.scaled_dot_product_attention(
            queries, keys, values, is_causal=True, dropout_p=dropout_p
        ).transpose(1, 2)

        # Combine heads, where self.dim_out = self.num_heads * self.head_dim
        out_embeds = out_embeds.reshape(batch_size, num_tokens, self.dim_out)
        out_embeds = self.out_proj(out_embeds)  # optional projection

        return out_embeds


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        # Constant kept as a Python float so nothing is re-materialized on the
        # host each call (and no CPU/CUDA tensor mismatch under torch.compile).
        return (
            0.5
            * x
            * (
                1
                + torch.tanh(
                    math.sqrt(2.0 / math.pi) * (x + 0.044715 * torch.pow(x, 3))
                )
            )
        )


class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(config.emb_dim, 4 * config.emb_dim),
            GELU(),
            nn.Linear(4 * config.emb_dim, config.emb_dim),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.att = MultiHeadAttention(
            dim_in=config.emb_dim,
            dim_out=config.emb_dim,
            ctx_len=config.ctx_len,
            num_heads=config.n_heads,
            dropout=config.drop_rate,
            bias=config.bias,
        )
        self.ff = FeedForward(config)
        self.norm1 = LayerNorm(config.emb_dim)
        self.norm2 = LayerNorm(config.emb_dim)
        self.drop_attn = nn.Dropout(config.drop_rate)
        self.drop_ff = nn.Dropout(config.drop_rate)

    def forward(self, x):
        # Attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_attn(x)
        x = x + shortcut  # Add the original input back

        # Feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_ff(x)
        x = x + shortcut  # Add the original input back

        return x


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.tok_emb = nn.Embedding(config.vocab_size, config.emb_dim)
        self.pos_emb = nn.Embedding(config.ctx_len, config.emb_dim)
        self.drop_emb = nn.Dropout(config.drop_rate)

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(config) for _ in range(config.n_layers)]
        )

        self.final_norm = LayerNorm(config.emb_dim)
        self.out_head = nn.Linear(config.emb_dim, config.vocab_size, bias=config.bias)
        # Weight tying (GPT-2 style): input and output embeddings share one matrix,
        # which is why nn.Linear's [vocab_size, emb_dim] weight lines up with
        # nn.Embedding's. Roughly halves the embedding parameter count.
        self.out_head.weight = self.tok_emb.weight

        # nn.Embedding's default init has std=1.0; left alone, the tied output
        # projection above would blow up logit magnitude (loss >> ln(vocab_size)
        # at init). GPT-2-style N(0, 0.02) keeps init sane for both roles.
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)

        return logits


def unwrap_model(model):
    if isinstance(model, DistributedDataParallel):
        model = model.module
    # torch.compile's wrapper prefixes state_dict keys with "_orig_mod.";
    # unwrap it too so checkpoints are portable across runs.
    return getattr(model, "_orig_mod", model)


def save_checkpoint(
    path,
    model,
    optimizer,
    step: int,
    gpt_config: GPTConfig,
    tokenizer_state: TokenizerState,
    train_losses: list[float],
    val_losses: list[float],
    tokens_seen: list[int],
) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)

    rng_state: dict[str, object] = {
        "cpu": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all(),
    }

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "step": step,
            "rng_state": rng_state,
            "gpt_config": asdict(gpt_config),
            "tokenizer_state": asdict(tokenizer_state),
            "train_losses": train_losses,
            "val_losses": val_losses,
            "tokens_seen": tokens_seen,
        },
        path,
    )

    if wandb.run is not None:
        # A single artifact name (rather than one per step) means "latest"
        # always resolves to the most recent checkpoint, which is what lets a
        # fresh environment (e.g. a new Kaggle session) resume from wandb.
        artifact = wandb.Artifact("checkpoint", type="model")
        artifact.add_file(path)
        wandb.log_artifact(artifact, aliases=[f"step-{step}"])


def load_checkpoint(path, model, optimizer, device):
    checkpoint = torch.load(path, map_location=device, weights_only=False)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    rng_state = checkpoint["rng_state"]
    torch.set_rng_state(rng_state["cpu"].cpu())
    torch.cuda.set_rng_state_all(rng_state["cuda"])

    return (
        checkpoint["step"],
        checkpoint["train_losses"],
        checkpoint["val_losses"],
        checkpoint["tokens_seen"],
    )


def download_latest_wandb_checkpoint(path: str) -> bool:
    # Best-effort fallback for resume_if_available: a fresh environment (e.g.
    # a new Kaggle session) has no local checkpoint, but wandb might have one
    # from a prior run. Returns False (rather than raising) on any failure so
    # a missing/misconfigured wandb project just falls through to a cold start.
    project = os.getenv("WANDB_PROJECT", "femtochat")
    try:
        api = wandb.Api()
        entity = os.getenv("WANDB_ENTITY") or api.default_entity
        artifact = api.artifact(f"{entity}/{project}/checkpoint:latest", type="model")
        artifact_dir = artifact.download()
    except Exception:
        return False

    downloaded_path = next(Path(artifact_dir).glob("*.pt"), None)
    if downloaded_path is None:
        return False

    Path(path).parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(downloaded_path), path)
    return True


def make_scaler(device: torch.device):
    return torch.amp.GradScaler(device=device.type, enabled=USE_AMP)


def ddp_setup() -> tuple[int, int, torch.device]:
    world_size = int(os.environ.get("WORLD_SIZE", "1"))

    if world_size > 1:
        # Launched via torchrun with more than one process: real multi-GPU DDP.
        rank = int(os.environ["RANK"])
        local_rank = int(os.environ["LOCAL_RANK"])
        dist.init_process_group(backend="nccl", rank=rank, world_size=world_size)
        torch.cuda.set_device(local_rank)
        return rank, world_size, torch.device(f"cuda:{local_rank}")

    # Single process: no process group and no DDP wrapping needed (Kaggle's
    # single-GPU case).
    return 0, 1, torch.device(DEVICE_TYPE)


def ddp_cleanup() -> None:
    if dist.is_initialized():
        dist.destroy_process_group()


def ddp_is_main(rank: int) -> bool:
    return rank == 0


def sync_context(model, is_last_microstep: bool):
    # Only all-reduce gradients on the final micro-step of an accumulation
    # window; a non-DDP model has no gradient sync to skip in the first place.
    if is_last_microstep or not isinstance(model, DistributedDataParallel):
        return nullcontext()
    return model.no_sync()


def repeat_loader(loader, skip_batches: int = 0):
    # skip_batches fast-forwards the stream on resume so we don't re-serve the
    # earliest windows. Complete epochs are skipped for free by advancing the
    # epoch counter.
    batches_per_epoch = len(loader)
    epoch = skip_batches // batches_per_epoch if skip_batches else 0
    remainder = skip_batches % batches_per_epoch if skip_batches else 0
    while True:
        # DistributedSampler reshuffles by epoch number, not by re-iterating.
        if hasattr(loader.sampler, "set_epoch"):
            loader.sampler.set_epoch(epoch)
        for batch_index, batch in enumerate(loader):
            if batch_index < remainder:
                continue
            yield batch
        remainder = 0  # only the resumed epoch is partially skipped
        epoch += 1


def get_lr(step: int) -> float:
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    if step > MAX_STEPS:
        return MIN_LR
    decay_ratio = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    coeff = 0.5 * (1 + math.cos(math.pi * decay_ratio))
    return MIN_LR + coeff * (LR - MIN_LR)


def train_micro_batch(
    model, scaler, input_batch, target_batch, device, grad_accum_steps
) -> float:
    model.train()
    with torch.autocast(
        device_type=device.type, dtype=AMP_DTYPE, enabled=scaler.is_enabled()
    ):
        logits = model(input_batch.to(device, non_blocking=True))
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            target_batch.to(device, non_blocking=True).reshape(-1),
        )
    scaler.scale(loss / grad_accum_steps).backward()

    return loss.item()


def evaluate(model, scaler, val_loader, device, eval_iters: int) -> float:
    model.eval()
    val_loss = 0.0
    num_batches = min(eval_iters, len(val_loader))
    with torch.no_grad():
        for i, (val_input, val_target) in enumerate(val_loader):
            if i >= num_batches:
                break
            # Mirror training precision so eval loss is measured on the same
            # numerics the model is optimized under.
            with torch.autocast(
                device_type=device.type, dtype=AMP_DTYPE, enabled=scaler.is_enabled()
            ):
                val_logits = model(val_input.to(device, non_blocking=True))
                batch_val_loss = F.cross_entropy(
                    val_logits.reshape(-1, val_logits.size(-1)),
                    val_target.to(device, non_blocking=True).reshape(-1),
                ).item()
            val_loss += batch_val_loss
    return val_loss / num_batches


def compute_update_ratios(model, lr):
    # (lr * grad_norm) / param_norm per layer: the classic "how big is this
    # step relative to the weight itself" diagnostic (Karpathy's ~1e-3 rule of
    # thumb) for spotting dead/exploding layers early in training.
    base = unwrap_model(model)
    named = {
        "tok_emb": base.tok_emb.weight,
        "block_0": base.trf_blocks[0].ff.layers[0].weight,
        "block_last": base.trf_blocks[-1].ff.layers[0].weight,
    }
    if base.out_head.weight is not base.tok_emb.weight:
        named["out_head"] = base.out_head.weight

    ratios = {}
    for name, param in named.items():
        if param.grad is None:
            continue
        ratio = (lr * param.grad.norm()) / (param.data.norm() + 1e-12)
        ratios[f"update_ratio/{name}"] = ratio.item()
    return ratios


def run_optimizer_step(model, optimizer, scaler, train_iter, device, step):
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    optimizer.zero_grad()
    accumulated_loss = 0.0
    step_tokens = 0
    for micro_step in range(GRAD_ACCUM_STEPS):
        input_batch, target_batch = next(train_iter)
        is_last_microstep = micro_step == GRAD_ACCUM_STEPS - 1
        with sync_context(model, is_last_microstep):
            accumulated_loss += train_micro_batch(
                model, scaler, input_batch, target_batch, device, GRAD_ACCUM_STEPS
            )
        step_tokens += input_batch.numel()
    accumulated_loss /= GRAD_ACCUM_STEPS

    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    update_ratios = compute_update_ratios(model, lr)
    scaler.step(optimizer)
    scaler.update()

    return accumulated_loss, step_tokens, lr, grad_norm, update_ratios


def evaluate_and_checkpoint(
    model,
    optimizer,
    scaler,
    val_loader,
    encoding,
    eot_token_id,
    gpt_config: GPTConfig,
    device: torch.device,
    tokenizer_state: TokenizerState,
    checkpoint_path: str,
    step_num: int,
    train_loss: float,
    total_tokens: int,
    train_losses: list[float],
    val_losses: list[float],
    tokens_seen: list[int],
    use_wandb: bool,
) -> tuple[float, str]:
    train_losses.append(train_loss)
    tokens_seen.append(total_tokens)

    val_loss = evaluate(model, scaler, val_loader, device, EVAL_ITERS)
    val_losses.append(val_loss)

    output_text = generate_text(
        model=model,
        encoding=encoding,
        eot_token_id=eot_token_id,
        prompt=VAL_PROMPT,
        max_tokens=MAX_SAMPLE_TOKENS,
        ctx_len=gpt_config.ctx_len,
        device=device,
        temperature=TEMPERATURE,
        top_k=TOP_K,
    )

    if use_wandb:
        wandb.log({"val/loss": val_loss, "val/sample": output_text}, step=step_num)

    save_checkpoint(
        checkpoint_path,
        model=unwrap_model(model),
        optimizer=optimizer,
        step=step_num,
        gpt_config=gpt_config,
        tokenizer_state=tokenizer_state,
        train_losses=train_losses,
        val_losses=val_losses,
        tokens_seen=tokens_seen,
    )

    return val_loss, output_text


def run_training(
    model,
    optimizer,
    scaler,
    train_loader,
    val_loader,
    encoding,
    eot_token_id,
    gpt_config: GPTConfig,
    device: torch.device,
    tokenizer_state: TokenizerState,
    checkpoint_path: str,
    is_main: bool,
    world_size: int = 1,
    use_wandb: bool = False,
    start_step: int = 0,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    tokens_seen: list[int] | None = None,
):
    train_losses = train_losses if train_losses is not None else []
    val_losses = val_losses if val_losses is not None else []
    tokens_seen = tokens_seen if tokens_seen is not None else []
    total_tokens = tokens_seen[-1] if tokens_seen else 0
    train_iter = repeat_loader(train_loader, skip_batches=start_step * GRAD_ACCUM_STEPS)

    for step in range(start_step, MAX_STEPS):
        accumulated_loss, step_tokens, lr, grad_norm, update_ratios = run_optimizer_step(
            model, optimizer, scaler, train_iter, device, step
        )
        total_tokens += step_tokens * world_size
        step_num = step + 1

        if is_main and use_wandb:
            wandb.log(
                {
                    "train/loss": accumulated_loss,
                    "train/lr": lr,
                    "train/grad_norm": grad_norm,
                    "train/tokens_seen": total_tokens,
                    "train/progress": step_num / MAX_STEPS * 100,
                    **update_ratios,
                },
                step=step_num,
            )

        if is_main and (step_num % EVAL_FREQ == 0 or step_num == MAX_STEPS):
            evaluate_and_checkpoint(
                model=model,
                optimizer=optimizer,
                scaler=scaler,
                val_loader=val_loader,
                encoding=encoding,
                eot_token_id=eot_token_id,
                gpt_config=gpt_config,
                device=device,
                tokenizer_state=tokenizer_state,
                checkpoint_path=checkpoint_path,
                step_num=step_num,
                train_loss=accumulated_loss,
                total_tokens=total_tokens,
                train_losses=train_losses,
                val_losses=val_losses,
                tokens_seen=tokens_seen,
                use_wandb=use_wandb,
            )

    return train_losses, val_losses, tokens_seen


def sample_ids(
    model, ids, eot_token_id, max_tokens, ctx_len, temperature=0.0, top_k=None
):
    for _ in range(max_tokens):
        # Crop current context if it exceeds the supported context size
        ids_cond = ids[:, -ctx_len:]
        with torch.no_grad():
            logits = model(ids_cond)

        logits = logits[:, -1, :]

        if temperature > 0:
            if top_k is not None:
                top_values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < top_values[:, [-1]]] = -float("inf")
            probs = F.softmax(logits / temperature, dim=-1)
            id_next = torch.multinomial(probs, num_samples=1)  # (batch, 1)
        else:
            id_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch, 1)

        ids = torch.cat((ids, id_next), dim=1)  # (batch, n_tokens+1)

        # Early stop only makes sense for a single sequence; for a batch,
        # different rows hit EOT at different steps, so keep generating.
        if ids.size(0) == 1 and id_next.item() == eot_token_id:
            break

    return ids


def generate_text(
    model,
    encoding,
    eot_token_id,
    prompt,
    max_tokens,
    ctx_len,
    device,
    temperature=0.0,
    top_k=None,
):
    model.eval()
    input_ids = torch.tensor(
        [eot_token_id] + encoding.encode_ordinary(prompt)
    ).unsqueeze(0)

    output_ids = sample_ids(
        model=model,
        ids=input_ids.to(device),
        eot_token_id=eot_token_id,
        max_tokens=max_tokens,
        ctx_len=ctx_len,
        temperature=temperature,
        top_k=top_k,
    )

    return encoding.decode(output_ids.squeeze(0).tolist())


def init_wandb(model, gpt_config: GPTConfig) -> None:
    wandb.init(
        project=os.getenv("WANDB_PROJECT", "femtochat"),
        entity=os.getenv("WANDB_ENTITY"),
        name=f"femtochat-{DEVICE_TYPE}-{PROFILE}",
        config={
            **asdict(gpt_config),
            "lr": LR,
            "batch_size": BATCH_SIZE,
            "grad_accum_steps": GRAD_ACCUM_STEPS,
            "max_steps": MAX_STEPS,
            "warmup_steps": WARMUP_STEPS,
            "weight_decay": WEIGHT_DECAY,
            "device_type": DEVICE_TYPE,
            "profile": PROFILE,
        },
    )
    wandb.watch(model, log="gradients", log_freq=EVAL_FREQ)


def build_dataloaders(
    train_ids, val_ids, gpt_config: GPTConfig, rank: int, world_size: int
):
    train_loader = make_dataloader(
        train_ids,
        batch_size=BATCH_SIZE,
        max_length=gpt_config.ctx_len,
        stride=gpt_config.ctx_len,
        shuffle=True,
        drop_last=True,
        rank=rank,
        world_size=world_size,
    )
    # Validation stays unsharded: only is_main evaluates, and it needs the
    # whole val set, not just this rank's slice.
    val_loader = make_dataloader(
        val_ids,
        batch_size=BATCH_SIZE,
        max_length=gpt_config.ctx_len,
        stride=gpt_config.ctx_len,
        shuffle=False,
        drop_last=False,
    )
    return train_loader, val_loader


def resume_if_available(model, optimizer, device, is_main: bool, world_size: int = 1):
    if RESUME_PATH is None:
        return 0, [], [], []

    if not os.path.exists(RESUME_PATH):
        # No local checkpoint -- try wandb before giving up. Only the main
        # rank hits the network; other ranks wait at the barrier and then
        # see the file main just wrote.
        if is_main and download_latest_wandb_checkpoint(RESUME_PATH):
            print(f"No local checkpoint; downloaded latest from wandb to {RESUME_PATH}")
        if world_size > 1:
            dist.barrier()
        if not os.path.exists(RESUME_PATH):
            return 0, [], [], []

    start_step, train_losses, val_losses, tokens_seen = load_checkpoint(
        RESUME_PATH, model, optimizer, device
    )
    if is_main:
        print(f"Resumed from {RESUME_PATH} at step {start_step}")
    return start_step, train_losses, val_losses, tokens_seen


def main() -> None:
    rank, world_size, device = ddp_setup()
    is_main = ddp_is_main(rank)
    use_wandb = is_main
    torch.manual_seed(SEED + rank)

    # Each rank is its own process with its own rapidgzip handle, so split
    # cores across ranks rather than each rank claiming all of them.
    gzip_parallelization = max(1, os.cpu_count() // world_size)
    if is_main:
        print(f"Using {device} device (world_size={world_size}, profile={PROFILE}).")

    with contextlib.ExitStack() as stack:
        # LIFO teardown: registered here, so cleanup runs bottom-up -- handles
        # close, then wandb.finish, then ddp_cleanup last.
        stack.callback(ddp_cleanup)
        if use_wandb:
            stack.callback(wandb.finish)

        encoding, eot_token_id, tokenizer_state = load_tokenizer_state(
            TOKENIZER_STATE_PATH
        )

        train_ids = GzipTokens(
            TRAIN_TOKENS_GZ_PATH,
            TRAIN_TOKENS_INDEX_PATH,
            parallelization=gzip_parallelization,
        )
        stack.callback(train_ids.close)
        val_ids = GzipTokens(
            VAL_TOKENS_GZ_PATH,
            VAL_TOKENS_INDEX_PATH,
            parallelization=gzip_parallelization,
        )
        stack.callback(val_ids.close)

        gpt_config = GPTConfig(vocab_size=encoding.n_vocab)
        model = GPT(gpt_config).to(device)

        if use_wandb:
            init_wandb(model, gpt_config)

        optimizer = torch.optim.AdamW(
            model.parameters(), lr=LR, betas=BETAS, weight_decay=WEIGHT_DECAY
        )

        # Resume before compile/DDP wrapping
        start_step, train_losses, val_losses, tokens_seen = resume_if_available(
            model, optimizer, device, is_main, world_size=world_size
        )

        # Compile before DDP so unwrap_model sees DDP(OptimizedModule(GPT)):
        # .module -> OptimizedModule -> _orig_mod -> GPT.
        if USE_COMPILE:
            model = torch.compile(model, mode=COMPILE_MODE)
        if world_size > 1:
            model = DistributedDataParallel(model, device_ids=[device.index])

        scaler = make_scaler(device)
        train_loader, val_loader = build_dataloaders(
            train_ids, val_ids, gpt_config, rank, world_size
        )

        run_training(
            model,
            optimizer,
            scaler,
            train_loader,
            val_loader,
            encoding,
            eot_token_id,
            gpt_config,
            device,
            tokenizer_state,
            CHECKPOINT_PATH,
            is_main,
            world_size=world_size,
            use_wandb=use_wandb,
            start_step=start_step,
            train_losses=train_losses,
            val_losses=val_losses,
            tokens_seen=tokens_seen,
        )

        if is_main:
            print(f"Saved checkpoint to {CHECKPOINT_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
if NUM_PROCS > 1:
    ! torchrun --standalone --nproc_per_node={NUM_PROCS} {TRAIN_SCRIPT_PATH}
else:
    ! python {TRAIN_SCRIPT_PATH}